# Druga kontrolna tačka: Veštačka neuronska mreža (ANN) od nule
**Studenti:** Luka Savkov (E2 63/2025), Stefan Bogdanović (R2 40/2024)  
**Predmet:** Sistemi za istraživanje i analizu podataka  
**Tema:** Regresija ocena (Rating) na osnovu metapodataka sa Google Play prodavnice.

---
### Cilj druge kontrolne tačke
Cilj ove faze je prevazilaženje plafona prethodnog *Random Forest* modela implementacijom potpuno prilagođene (custom) Veštačke neuronske mreže, izgrađene od nule (eng. *from scratch*). Pored kreiranja same arhitekture mreže, fokus je bio na uvođenju naprednih koncepata inženjeringa obeležja (Feature Engineering) kako bismo iz sirovih podataka izvukli maksimum konteksta.

## 1. Napredni inženjering obeležja (Feature Engineering)
Da bismo mreži dali dublji kontekst, u odnosu na prvu kontrolnu tačku uveli smo sledeća poboljšanja u pretprocesiranju:
1. **Ekstrakcija vremenskih podataka:** Izračunali smo `Days_Since_Last_Update` ukrštanjem datuma poslednjeg ažuriranja i datuma prikupljanja podataka kako bismo utvrdili "zapuštenost" aplikacije.
2. **Dužina imena aplikacije:** Izbrojali smo karaktere originalnog imena (`Name_Length`) i prosledili ih mreži.
3. **Logaritmovanje ekstrema:** Kolone sa ogromnim varijacijama (`Rating Count`, `Minimum Installs`, `Price`) su transformisane logaritamskom funkcijom (`log1p`).
4. **Standardizacija (Z-score):** Svi ulazni podaci su normalizovani oko nule kako bi se stabilizovali gradijenti unutar skrivenih slojeva mreže.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Importovanje sopstvenih klasa za neuronsku mrežu
from ann.dense import Dense
from ann.train import train, predict
from ann.helper import standardize
from ann.batch_norm import BatchNorm
from ann.dropout import Dropout

file_name = 'Google-Playstore.csv' 

df = pd.read_csv(file_name)
df = df[df['Rating Count'] > 0].copy()
df.dropna(subset=['Rating'], inplace=True)

def clean_size(size_val):
    size_str = str(size_val).replace(',', '')
    if 'M' in size_str:
        return float(size_str.replace('M', ''))
    elif 'k' in size_str:
        return float(size_str.replace('k', '')) / 1024
    elif 'G' in size_str:
        return float(size_str.replace('G', '')) * 1024
    elif 'Varies with device' in size_str:
        return np.nan
    else:
        try:
            return float(size_str)
        except:
            return np.nan

df['Size_MB'] = df['Size'].apply(clean_size)
df['Size_MB'] = df['Size_MB'].fillna(df['Size_MB'].mean())

features = [
    'App Name', 'Category', 'Size_MB', 'Minimum Installs', 'Price', 
    'Content Rating', 'Ad Supported', 'In App Purchases',
    'Editors Choice', 'Rating Count', 'Last Updated', 'Scraped Time'
]
target = 'Rating'
df_model = df[features + [target]].copy()
df_model.dropna(inplace=True)

# Pretprocesiranje
df_model['Name_Length'] = df_model['App Name'].astype(str).apply(len)
df_model.drop('App Name', axis=1, inplace=True)

df_model['Scraped Time'] = pd.to_datetime(df_model['Scraped Time'], format='mixed')
df_model['Last Updated'] = pd.to_datetime(df_model['Last Updated'], format='mixed')
df_model['Days_Since_Last_Update'] = (df_model['Scraped Time'] - df_model['Last Updated']).dt.days
df_model.drop(['Scraped Time', 'Last Updated'], axis=1, inplace=True)

df_model['Minimum Installs'] = np.log1p(df_model['Minimum Installs'])
df_model['Price'] = np.log1p(df_model['Price'])
df_model['Rating Count'] = np.log1p(df_model['Rating Count'])

df_model['Size_Category'] = pd.qcut(df_model['Size_MB'], q=5, labels=['Very_Small', 'Small', 'Medium', 'Large', 'Very_Large'])
df_model.drop('Size_MB', axis=1, inplace=True)

df_model['Ad Supported'] = df_model['Ad Supported'].astype(int)
df_model['In App Purchases'] = df_model['In App Purchases'].astype(int)
df_model['Editors Choice'] = df_model['Editors Choice'].astype(int)

df_model = pd.get_dummies(df_model, columns=['Category', 'Content Rating', 'Size_Category'], drop_first=True, dtype=int)

final_features = [col for col in df_model.columns if col != target]
X = df_model[final_features]
y = df_model[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ANN Arhitektura
X_ann_train_scaled, mu, std = standardize(X_train.values)
X_ann_test_scaled, _, _ = standardize(X_test.values, mu, std)
y_ann_train = y_train.values.reshape(-1, 1)
y_ann_test = y_test.values.reshape(-1, 1)

layers = [
    Dense(X_ann_train_scaled.shape[1], 128, activation='parametric_relu', optimizer_type='adam'), 
    BatchNorm(),
    Dense(128, 64, activation='parametric_relu', optimizer_type='adam'),
    Dropout(p=0.1),
    Dense(64, 16, activation='parametric_relu', optimizer_type='adam'),
    Dense(16, 1, activation='linear', optimizer_type='adam')
]

print("Treniranje Custom ANN modela u toku...")
loss_history = train(X_ann_train_scaled, y_ann_train, layers, epochs=60, learning_rate=0.001, cost_type='mse', lr_decay='step_decay', batch_size=2048, D=15, F=0.8)

y_ann_pred = predict(X_ann_test_scaled, layers)

print(f"MAE:  {mean_absolute_error(y_ann_test, y_ann_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_ann_test, y_ann_pred)):.4f}")
print(f"R2:   {r2_score(y_ann_test, y_ann_pred):.4f}")

Treniranje Custom ANN modela u toku...
Epoch  1/60  |  lr = 0.001000  |  loss = 0.379485
Epoch 10/60  |  lr = 0.001000  |  loss = 0.193841
Epoch 20/60  |  lr = 0.000800  |  loss = 0.189429
Epoch 30/60  |  lr = 0.000640  |  loss = 0.187736
Epoch 40/60  |  lr = 0.000640  |  loss = 0.186826
Epoch 50/60  |  lr = 0.000512  |  loss = 0.186040
Epoch 60/60  |  lr = 0.000410  |  loss = 0.185429
MAE:  0.4526
RMSE: 0.6291
R2:   0.1674


## 2. Uporedna analiza rezultata (Random Forest vs Custom ANN)
Nakon obučavanja modela i provere na test skupu (koji čini 20% podataka), dobijeni rezultati nedvosmisleno pokazuju superiornost prilagođene mrežne arhitekture u odnosu na naš bazni model.

| Metrika | Random Forest (1. Kontrolna) | Custom ANN (2. Kontrolna) | Napredak |
| :--- | :--- | :--- | :--- |
| **MAE** | ~0.4930 | **0.4526** | Probijena granica od pola zvezdice |
| **RMSE** | ~0.6616 | **0.6291** | Značajno smanjen uticaj autlajera |
| **R2 Score** | ~0.0818 | **0.1674** | Više nego duplirana objašnjena varijansa |

## 3. Arhitektura i Zaključak
Naša mreža je dizajnirana po principu "levka" (ekspanzija na 128, pa kompresija na 64 i 16 neurona) kako bi postepeno učila složene hijerarhijske koncepte i izvlačila suštinske obrasce iz visoko-dimenzionalnih (63 kolone) One-Hot enkodiranih podataka.

**Ključni tehnički detalji arhitekture:**
* **Optimizator:** Koristili smo *Adam (Adaptive Moment Estimation)* kako bismo omogućili nezavisno prilagođavanje brzine učenja za svaki parametar, uz *Step Decay* tehniku koja je sprečila preskakanje globalnog minimuma.
* **Aktivacione funkcije:** *Parametric ReLU (PReLU)* u skrivenim slojevima (rešava problem umirućih neurona izazvan Z-score standardizacijom) i *Linear* na izlazu za potrebe regresije.
* **Regularizacija:** *Batch Normalization* je implementiran nakon prvog, najvećeg sloja za stabilizaciju izlaza (borba protiv Internal Covariate Shift-a), dok je *Dropout (10%)* postavljen kako bi mreža gradila robusnije zaključke umesto učenja tabele napamet.

**Zaključak:** Kroz rigorozno definisanje custom arhitekture i naprednu obradu parametara, uspešno smo probili takozvani plafon podataka, nadmašili moćan ansambl (Random Forest) model i time dokazali uspeh dizajna ove mreže na izuzetno subjektivnom regresionom problemu.